In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# ADIM 1: DOSYALARI OKUMA (Klasör yapına göre güncellendi)
try:
    # ml.ipynb dosyası financial_data içindeyse, ../ ile bir üst klasöre çıkar
    watch_df = pd.read_csv('../saat_verileri_son.csv')
    finance_df = pd.read_csv('../finance.csv')
    
    finance_pivoted = finance_df.pivot(index='Year', columns='Indicator_Name', values='Value').reset_index()
    df_master = pd.merge(watch_df, finance_pivoted, on='Year', how='left')
    df_master = df_master.fillna(df_master.mean(numeric_only=True))
    print("✅ Veriler başarıyla birleştirildi.")

    # ADIM 2: K-MEANS
    cluster_features = ['Value', 'Gold_USD', 'S&P500_Index', 'US_CPI_Index']
    X_cluster = df_master[cluster_features]
    X_scaled = StandardScaler().fit_transform(X_cluster)
    df_master['Cluster'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)

    # GRAFİK 1
    plt.figure(figsize=(8, 4))
    sns.scatterplot(data=df_master, x='Gold_USD', y='Value', hue='Cluster', palette='Set1')
    plt.title('Saat Grupları')
    plt.show()

    # ADIM 3: RANDOM FOREST
    features = ['Year', 'Gold_USD', 'Nasdaq_Index', 'S&P500_Index', 'US_CPI_Index', 'Cluster']
    X = df_master[features]
    y = df_master['Value']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

    print(f"📊 Model Başarı Skoru (R2): {r2_score(y_test, rf.predict(X_test)):.4f}")

    # GRAFİK 2
    feat_importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)
    plt.figure(figsize=(8, 4))
    feat_importances.plot(kind='barh')
    plt.title('Fiyatı Etkileyen Faktörler')
    plt.show()

except Exception as e:
    print(f"❌ Hata: {e}")